# 1 · Dados, pré-processamento e detecção

Implementação da proposta para **imagens estáticas**. O export Roboflow Samu é a fonte principal; o experimento separa grupos antes do treino. Execute as células em ordem. Resultados, dados preparados e pesos ficam fora do Git.

O export não informa a identidade física dos componentes ou de todas as cenas. O agrupamento por nome original e cópia idêntica reduz vazamentos conhecidos, mas não garante independência de cenas ou objetos; isso exige revisão das anotações.

In [ ]:
from pathlib import Path
import os, json, hashlib, random, re, shutil, time
from collections import Counter, defaultdict
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, yaml
from ultralytics import YOLO
def read_image(path):
    try: return cv2.imdecode(np.fromfile(str(path),dtype=np.uint8),cv2.IMREAD_COLOR)
    except (OSError,cv2.error): return None

DATA = Path(os.environ.get("RESISTOR_DATA_ROOT", "../data")).expanduser().resolve()
OUTPUT = Path(os.environ.get("RESISTOR_OUTPUT", "../outputs/proposta")).expanduser().resolve()
SOURCE = DATA / "sources/roboflow-electronic-components/extracted"
PREPARED = OUTPUT / "samu-agrupado"
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
assert SOURCE.is_dir(), f"Dataset não encontrado: {SOURCE}. Configure RESISTOR_DATA_ROOT."
print("Dataset:", SOURCE, "\nResultados:", OUTPUT)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Inventário e divisão

Agrupamos pelo nome anterior ao sufixo Roboflow `.rf.` e também por SHA-256. Depois dividimos os grupos em aproximadamente 70/15/15. A classe original **Resistor** (5) é remapeada para 0; as outras classes são ignoradas. Imagens sem resistor permanecem como exemplos negativos.

A divisão é nova: checkpoints treinados na divisão antiga **não** são usados na avaliação desta divisão.

In [ ]:
images = sorted(p for split in ("train", "valid", "test") for p in (SOURCE / split / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
assert images, "Nenhuma imagem encontrada"
parent = list(range(len(images)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(a, b):
    a, b = find(a), find(b)
    if a != b: parent[b] = a
by_name, by_hash = {}, {}
for i, p in enumerate(images):
    name = re.split(r"\.rf\.", p.name, flags=re.I)[0].lower()
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    for seen, key in ((by_name, name), (by_hash, digest)):
        if key in seen: union(i, seen[key])
        else: seen[key] = i
groups = defaultdict(list)
for i in range(len(images)): groups[find(i)].append(i)
order = list(groups.values())
random.Random(SEED).shuffle(order)
# Grupos maiores primeiro para aproximar as proporções por imagem.
order.sort(key=len, reverse=True)
targets = {"train": .70 * len(images), "val": .15 * len(images), "test": .15 * len(images)}
assigned = {k: [] for k in targets}
for group in order:
    dest = max(targets, key=lambda k: (targets[k] - len(assigned[k])) / targets[k])
    assigned[dest].extend(group)
print("Export:", len(images), "imagens;", len(groups), "grupos")
print("Nova divisão:", {k: len(v) for k,v in assigned.items()})


In [ ]:
rows = []
for split, indexes in assigned.items():
    for i in indexes:
        image = images[i]
        label = image.parent.parent / "labels" / (image.stem + ".txt")
        if not label.exists():
            raise FileNotFoundError(f"Rótulo ausente: {label}")
        kept = []
        for line in label.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if parts and int(float(parts[0])) == 5:
                if len(parts) != 5: raise ValueError(f"Caixa inválida: {label}")
                kept.append("0 " + " ".join(parts[1:]))
        unique = hashlib.sha256(str(image.relative_to(SOURCE)).encode()).hexdigest()[:16]
        image_dir, label_dir = PREPARED / split / "images", PREPARED / split / "labels"
        image_dir.mkdir(parents=True, exist_ok=True); label_dir.mkdir(parents=True, exist_ok=True)
        target = image_dir / (unique + image.suffix.lower())
        if not target.exists():
            try: os.link(image, target)
            except OSError: shutil.copy2(image, target)
        (label_dir / (unique + ".txt")).write_text("\n".join(kept) + ("\n" if kept else ""), encoding="utf-8")
        rows.append({"id": unique, "origem": str(image.relative_to(SOURCE)), "grupo": find(i), "split": split, "resistores": len(kept)})
manifest = pd.DataFrame(rows)
manifest.to_csv(PREPARED / "manifesto.csv", index=False)
assert manifest.groupby("grupo")["split"].nunique().max() == 1
config = {"path": str(PREPARED), "train": "train/images", "val": "val/images", "test": "test/images", "names": {0: "resistor"}}
(PREPARED / "data.yaml").write_text(yaml.safe_dump(config, allow_unicode=True), encoding="utf-8")
print(manifest.groupby("split").agg(imagens=("id","size"), com_resistor=("resistores",lambda x:(x>0).sum()), caixas=("resistores","sum")))


## Iluminação, ruído e espaços de cor

A correção por CLAHE, suavização gaussiana e as representações RGB, HSV e CIELAB são exibidas nas mesmas imagens para comparação. O treinamento aplica variações de cor, rotação e escala somente no treino. A detecção clássica adiante usa bordas, morfologia e contornos.

In [ ]:
positive = manifest[(manifest.split == "train") & (manifest.resistores > 0)].iloc[0]
p = next((PREPARED / "train/images").glob(positive.id + ".*"))
bgr = read_image(p); assert bgr is not None
lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
corrected = cv2.cvtColor(cv2.merge((cv2.createCLAHE(2.0, (8,8)).apply(l), a, b)), cv2.COLOR_LAB2BGR)
blurred = cv2.GaussianBlur(corrected, (5,5), 0)
fig, axs = plt.subplots(1, 4, figsize=(15,4))
for ax, arr, title in zip(axs, [cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB), cv2.cvtColor(blurred,cv2.COLOR_BGR2RGB), cv2.cvtColor(blurred,cv2.COLOR_BGR2HSV), cv2.cvtColor(blurred,cv2.COLOR_BGR2LAB)], ["RGB original", "Correção + ruído", "Canais HSV", "Canais CIELAB"]):
    ax.imshow(arr); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


## Treino e localização

YOLO11n parte de pesos genéricos e é treinado exclusivamente com o novo treino. A validação seleciona o melhor checkpoint; o teste é usado somente na avaliação final. A configuração pode ser alterada por variáveis de ambiente para reprodução.

In [ ]:
epochs = int(os.environ.get("RESISTOR_EPOCHS", "30"))
batch = int(os.environ.get("RESISTOR_BATCH", "8"))
existing=os.environ.get("RESISTOR_EVALUATE_WEIGHTS")
if existing:
    best=Path(existing).expanduser().resolve()
    print("Reutilizando pesos informados; verifique se foram treinados apenas na divisão de treino.")
else:
    model = YOLO(os.environ.get("RESISTOR_INITIAL_WEIGHTS", "yolo11n.pt"))
    training = model.train(data=str(PREPARED / "data.yaml"), epochs=epochs, imgsz=640, batch=batch,
        device=0 if torch.cuda.is_available() else "cpu", workers=2, seed=SEED, deterministic=True,
        degrees=15, hsv_h=.015, hsv_s=.4, hsv_v=.3, fliplr=.5, mosaic=.5,
        project=str(OUTPUT / "treino"), name="yolo-samu", exist_ok=True, plots=True)
    best = OUTPUT / "treino/yolo-samu/weights/best.pt"
assert best.exists(), f"Checkpoint não encontrado: {best}"
print("Melhor checkpoint:", best)


## Avaliação no teste reservado

Precisão, recall e mAP vêm do avaliador Ultralytics. A comparação clássica usa as mesmas imagens e rótulos, IoU ≥ 0,5 e associação um a um. A contagem exata e o erro absoluto médio são por imagem, incluindo negativos. O tempo é medido por imagem após aquecimento.

In [ ]:
model = YOLO(str(best))
metrics = model.val(data=str(PREPARED / "data.yaml"), split="test", imgsz=640, batch=batch,
                    device=0 if torch.cuda.is_available() else "cpu", plots=False, project=str(OUTPUT / "avaliacao"), name="teste", exist_ok=True)
map_result = {"precision": float(metrics.box.mp), "recall": float(metrics.box.mr), "map50": float(metrics.box.map50), "map50_95": float(metrics.box.map)}
print(map_result)


In [ ]:
def classic_boxes(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 45, 135)
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((5,5),np.uint8), iterations=2)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if not 350 <= area <= image.shape[0]*image.shape[1]*.8: continue
        _, (w,h), _ = cv2.minAreaRect(contour)
        lo, hi = sorted((w,h))
        if lo < 8 or not 1.8 <= hi/lo <= 12: continue
        x,y,w,h = cv2.boundingRect(contour)
        result.append([x,y,x+w,y+h])
    return result

def iou(a,b):
    x1,y1,x2,y2 = max(a[0],b[0]),max(a[1],b[1]),min(a[2],b[2]),min(a[3],b[3])
    inter = max(0,x2-x1)*max(0,y2-y1)
    return inter/max(1e-9,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter)

def box_stats(gt,pred):
    choices = sorted(((iou(g,p), i,j) for i,g in enumerate(gt) for j,p in enumerate(pred)), reverse=True)
    matched_g, matched_p = set(),set()
    for score,i,j in choices:
        if score < .5: break
        if i not in matched_g and j not in matched_p: matched_g.add(i); matched_p.add(j)
    return len(matched_g),len(pred)-len(matched_p),len(gt)-len(matched_g)

def gt_boxes(path, width, height):
    result=[]
    for line in path.read_text(encoding="utf-8").splitlines():
        _,x,y,w,h = map(float,line.split())
        result.append([(x-w/2)*width,(y-h/2)*height,(x+w/2)*width,(y+h/2)*height])
    return result


In [ ]:
test_rows = manifest[manifest.split == "test"]
records = []
# Uma inferência de aquecimento não entra na medição.
first_path = next((PREPARED / "test/images").glob(test_rows.iloc[0].id + ".*"))
model.predict(read_image(first_path), conf=.25, verbose=False, device=0 if torch.cuda.is_available() else "cpu")
for row in test_rows.itertuples(index=False):
    path = next((PREPARED / "test/images").glob(row.id + ".*"))
    frame = read_image(path); assert frame is not None
    truth = gt_boxes(PREPARED / "test/labels" / (row.id + ".txt"), frame.shape[1], frame.shape[0])
    for method in ("classico", "yolo"):
        start = time.perf_counter()
        if method == "classico": pred = classic_boxes(frame)
        else:
            result = model.predict(frame, conf=.25, verbose=False, device=0 if torch.cuda.is_available() else "cpu")[0]
            pred = result.boxes.xyxy.cpu().numpy().tolist()
        elapsed = (time.perf_counter()-start)*1000
        tp,fp,fn = box_stats(truth,pred)
        records.append({"id":row.id,"metodo":method,"esperado":len(truth),"detectado":len(pred), "tp":tp,"fp":fp,"fn":fn,"ms":elapsed})
per_image = pd.DataFrame(records)
per_image.to_csv(OUTPUT / "deteccao_por_imagem.csv", index=False)
def summarize(g):
    tp,fp,fn = g.tp.sum(),g.fp.sum(),g.fn.sum()
    positive = g[g.esperado > 0]
    return pd.Series({"imagens":len(g), "imagens_positivas":len(positive), "contagem_exata":float((g.esperado==g.detectado).mean()), "contagem_exata_positivas":float((positive.esperado==positive.detectado).mean()) if len(positive) else float('nan'), "erro_absoluto_medio":float((g.esperado-g.detectado).abs().mean()), "precisao_iou50":tp/(tp+fp) if tp+fp else 0., "recall_iou50":tp/(tp+fn) if tp+fn else 0., "tempo_medio_ms":g.ms.mean()})
summary = per_image.groupby("metodo").apply(summarize, include_groups=False)
summary.to_csv(OUTPUT / "deteccao_resumo.csv")
(OUTPUT / "map_yolo.json").write_text(json.dumps(map_result,indent=2),encoding="utf-8")
display(summary)


A meta de 95% é uma **meta**, não um resultado garantido. Este teste não comprova desempenho em cenas ou componentes fisicamente novos: esses dados precisam ser identificados e separados na origem. O dataset Samu não contém rótulos de cores, valor ou tolerância; essas métricas são medidas no próximo notebook.